In [1]:
import os
import glob
import re
import pandas as pd

PRIMARY_PATH = "/kaggle/input/datasets/christiancorvino/cholec80-mr-sapienza"
FALLBACK_PATH = "/kaggle/input/cholec80-mr-sapienza"
DATASET_PATH = PRIMARY_PATH if os.path.exists(PRIMARY_PATH) else FALLBACK_PATH

def validazione_avanzata_cholec80(percorso_base):
    if not os.path.exists(percorso_base):
        print(f"❌ ERRORE: Percorso '{percorso_base}' non trovato.")
        return

    print(f"🔍 INIZIO VALIDAZIONE AVANZATA DATASET: {percorso_base}\n")
    print("=" * 60)

    # 1. ANALISI FILE CSV EXTRA NELLA ROOT
    print("\n📄 1. RICERCA FILE CSV GENERALI")
    csv_files = glob.glob(os.path.join(percorso_base, "*.csv"))
    if csv_files:
        for csv in csv_files:
            try:
                df_csv = pd.read_csv(csv)
                print(f"  * Trovato: {os.path.basename(csv)}")
                print(f"  * Dimensioni: {df_csv.shape[0]} righe x {df_csv.shape[1]} colonne")
                print(f"  * Colonne: {list(df_csv.columns)}")
            except Exception as e:
                print(f"  * ⚠️ Errore lettura {os.path.basename(csv)}: {e}")
    else:
        print("  * Nessun file .csv generico trovato nella cartella principale.")

    print("\n" + "=" * 60)

    # 2. CONTROLLO ESISTENZA E CORRISPONDENZA FILE (CROSS-CHECK)
    print("\n🔗 2. CROSS-CHECK CORRISPONDENZE (Video <-> Fasi <-> Strumenti)")
    videos_path = os.path.join(percorso_base, "videos")
    phases_path = os.path.join(percorso_base, "phase_annotations")
    tools_path = os.path.join(percorso_base, "tool_annotations")

    if os.path.exists(videos_path) and os.path.exists(phases_path) and os.path.exists(tools_path):
        video_files = [f for f in os.listdir(videos_path) if f.endswith('.mp4')]
        
        # Estraiamo i numeri ID dei video (es. da "video28.mp4" estraiamo "28")
        video_ids = set(re.findall(r'\d+', f)[0] for f in video_files if re.findall(r'\d+', f))
        
        missing_phases = []
        missing_tools = []
        
        for vid in video_ids:
            if not os.path.exists(os.path.join(phases_path, f"video{vid}-phase.txt")):
                missing_phases.append(vid)
            if not os.path.exists(os.path.join(tools_path, f"video{vid}-tool.txt")):
                missing_tools.append(vid)
                
        if not missing_phases and not missing_tools:
            print(f"  * ✅ PERFETTO: Tutti i {len(video_ids)} video hanno le corrispondenti annotazioni di fase e tool.")
        else:
            if missing_phases:
                print(f"  * ⚠️ ATTENZIONE: Fasi mancanti per i video: {missing_phases}")
            if missing_tools:
                print(f"  * ⚠️ ATTENZIONE: Tool mancanti per i video: {missing_tools}")
    else:
        print("  * ⚠️ Impossibile eseguire il cross-check: cartelle mancanti.")

    print("\n" + "=" * 60)

    # 3. ANALISI BILANCIAMENTO CLASSI (PHASE & TOOLS)
    print("\n⚖️ 3. ANALISI BILANCIAMENTO DATASET")
    
    # Bilanciamento Fasi
    if os.path.exists(phases_path):
        print("\n  [FASI CHIRURGHICHE]")
        phase_files = glob.glob(os.path.join(phases_path, "*.txt"))
        all_phases_counts = pd.Series(dtype=int)
        
        print("  * Lettura di tutte le annotazioni di fase in corso...")
        for pf in phase_files:
            try:
                # header=0 per saltare l'intestazione che a volte è scritta male
                df = pd.read_csv(pf, sep=r'\t|,|\s+', engine='python', header=0)
                # Assumiamo che la fase sia nell'ultima colonna o nella colonna chiamata 'Phase'
                phase_col = 'Phase' if 'Phase' in df.columns else df.columns[-1]
                counts = df[phase_col].value_counts()
                all_phases_counts = all_phases_counts.add(counts, fill_value=0)
            except:
                pass
                
        all_phases_counts = all_phases_counts.sort_values(ascending=False).astype(int)
        tot_frames = all_phases_counts.sum()
        
        for phase, count in all_phases_counts.items():
            percent = (count / tot_frames) * 100
            print(f"    - {phase}: {count} frames ({percent:.2f}%)")
            
    # Bilanciamento Strumenti
    if os.path.exists(tools_path):
        print("\n  [STRUMENTI (Presenza = 1)]")
        tool_files = glob.glob(os.path.join(tools_path, "*.txt"))
        
        # Colonne attese (escludendo 'Frame')
        tool_names = ["Grasper", "Bipolar", "Hook", "Scissors", "Clipper", "Irrigator", "Specimen_Bag", "SpecimenBag"]
        tool_sums = {}
        
        print("  * Lettura di tutte le annotazioni degli strumenti in corso...")
        tot_tool_frames = 0
        for tf in tool_files:
            try:
                df = pd.read_csv(tf, sep=r'\t|,|\s+', engine='python', header=0)
                tot_tool_frames += len(df)
                for col in df.columns:
                    if col in tool_names:
                        # Sommiamo gli 1 presenti nella colonna
                        tool_sums[col] = tool_sums.get(col, 0) + df[col].sum()
            except:
                pass
                
        # Accorpiamo eventuali nomi scritti diversamente (es. Specimen_Bag e SpecimenBag)
        if "SpecimenBag" in tool_sums and "Specimen_Bag" in tool_sums:
            tool_sums["Specimen_Bag"] += tool_sums.pop("SpecimenBag")
            
        tool_sums = dict(sorted(tool_sums.items(), key=lambda item: item[1], reverse=True))
        
        for tool, count in tool_sums.items():
            percent = (count / tot_tool_frames) * 100 if tot_tool_frames > 0 else 0
            print(f"    - {tool}: {int(count)} apparizioni ({percent:.2f}% dei frame annotati a 1 fps)")
            
    print("\n" + "=" * 60)
    print("🏁 VALIDAZIONE COMPLETATA.")

# Lancia l'analisi
validazione_avanzata_cholec80(DATASET_PATH)

🔍 INIZIO VALIDAZIONE AVANZATA DATASET: /kaggle/input/datasets/christiancorvino/cholec80-mr-sapienza


📄 1. RICERCA FILE CSV GENERALI
  * Nessun file .csv generico trovato nella cartella principale.


🔗 2. CROSS-CHECK CORRISPONDENZE (Video <-> Fasi <-> Strumenti)
  * ✅ PERFETTO: Tutti i 80 video hanno le corrispondenti annotazioni di fase e tool.


⚖️ 3. ANALISI BILANCIAMENTO DATASET

  [FASI CHIRURGHICHE]
  * Lettura di tutte le annotazioni di fase in corso...
    - CalotTriangleDissection: 1870651 frames (40.56%)
    - GallbladderDissection: 1460825 frames (31.67%)
    - CleaningCoagulation: 358303 frames (7.77%)
    - ClippingCutting: 352000 frames (7.63%)
    - Preparation: 214301 frames (4.65%)
    - GallbladderPackaging: 190450 frames (4.13%)
    - GallbladderRetraction: 166002 frames (3.60%)

  [STRUMENTI (Presenza = 1)]
  * Lettura di tutte le annotazioni degli strumenti in corso...
    - Hook: 103106 apparizioni (55.88% dei frame annotati a 1 fps)
    - Grasper: 102588 apparizi

### **1. Integrità Perfetta (Zero Pulizia Necessaria)**

* **Cross-Check superato:** Il dataset è immacolato. Come previsto dalle specifiche ufficiali, hai esattamente 80 video e tutte le 160 annotazioni corrispondenti (80 file per le fasi e 80 file per gli strumenti).


* **Nessun file orfano:** Non dovrai scartare alcun video per mancanza di etichette, il che ti fa risparmiare molto tempo di pre-processing.

### **2. Fasi Chirurgiche: Sbilanciamento Severo**

* **Classi Dominanti:** Oltre il **72%** del tempo chirurgico totale è concentrato in sole due fasi: *Calot Triangle Dissection* (40.56%) e *Gallbladder Dissection* (31.67%).
* **Classi Rare:** Le restanti cinque fasi si spartiscono le briciole. Fasi critiche come la *Preparation* (4.65%) o la *Gallbladder Retraction* (3.60%) sono rarissime.
* **Impatto sul Training:** Se addestri il modello "così com'è" usando una Loss standard (es. Cross-Entropy), la rete neurale tenderà a predire sempre le due fasi dominanti per abbassare l'errore, ignorando completamente le fasi più brevi. **Dovrai obbligatoriamente utilizzare tecniche di bilanciamento**, come una *Weighted Loss* (dando un peso molto più alto agli errori sulle classi rare) o tecniche di campionamento strategico nel tuo Dataloader.

### **3. Strumenti Chirurgici (Tool): Presenza Polarizzata**

* **I "Sempre Presenti":** L'*Hook* e il *Grasper* dominano la scena, essendo presenti in oltre il **55%** dei frame annotati.
* **Gli "Invisibili":** Strumenti come *Scissors* (1.76%) e *Clipper* (3.24%) appaiono quasi mai rispetto al totale.
* **Impatto sul Training:** Trattandosi di un task di classificazione multi-label (in un frame possono esserci zero, uno o più strumenti contemporaneamente), il modello farà un'enorme fatica a imparare i *feature patterns* delle forbici o del clipper. Anche qui, pesare le singole classi nella funzione di Loss sarà fondamentale.

In conclusione: la struttura dei file è pronta all'uso, ma i log confermano che la vera sfida ingegneristica per questo progetto sarà interamente incentrata su **come gestirai lo sbilanciamento dei dati** durante la fase di apprendimento.